# Train RF-DETR Colony Detector on AGAR Dataset (Kaggle)

This notebook trains an RF-DETR Medium detector using the **AGAR dataset** (`anilanna4678/agar-dataset`).

### Before Running (Right-hand Settings Panel):
- **Accelerator**: `GPU T4 x2` (or P100)
- **Internet**: `On`
- **Data Input**: Ensure `anilanna4678/agar-dataset` is attached in the **Input** section.

Then click **Save Version → Save & Run All (Commit)** or run interactively cell-by-cell.

In [ ]:
# 1. GPU Check & Setup Workspace
import torch, os, sys, shutil, glob
assert torch.cuda.is_available(), 'GPU not found! Enable GPU T4 in Settings panel.'
print('GPU Device:', torch.cuda.get_device_name(0))
OUT = '/kaggle/working'

# Clone colony_detector repository
!rm -rf /kaggle/temp/colony_detector && git clone -q https://github.com/akashbabu9392/colony_detector.git /kaggle/temp/colony_detector
%cd /kaggle/temp/colony_detector
!pip -q install -r requirements.txt "rfdetr[train]>=1.7" pyyaml
print('Setup complete.')

In [ ]:
# 2. Locate AGAR Dataset in /kaggle/input
candidates = glob.glob('/kaggle/input/**/dataset', recursive=True) + \
             glob.glob('/kaggle/input/**/raw', recursive=True) + \
             glob.glob('/kaggle/input/**/agar-dataset', recursive=True)

agar_dir = None
for c in candidates:
    if len(glob.glob(f'{c}/*.json')) > 0 or len(glob.glob(f'{c}/**/*.json', recursive=True)) > 0:
        agar_dir = c
        break

assert agar_dir is not None, f'AGAR dataset not found in /kaggle/input! Available inputs: {os.listdir("/kaggle/input")}'
print('Using AGAR dataset directory:', agar_dir)

jpgs = glob.glob(f'{agar_dir}/**/*.jpg', recursive=True) if len(glob.glob(f'{agar_dir}/*.jpg')) == 0 else glob.glob(f'{agar_dir}/*.jpg')
jsons = glob.glob(f'{agar_dir}/**/*.json', recursive=True) if len(glob.glob(f'{agar_dir}/*.json')) == 0 else glob.glob(f'{agar_dir}/*.json')
print(f'Detected: {len(jpgs)} images and {len(jsons)} annotation JSON files.')

In [ ]:
# 3. Select Balanced Countable Plates & Slice into Native Tiles
# Sample 1,400 countable plates (1-300 colonies) + 100 empty dishes (~20,000 tiles).
# This ensures 10 epochs runs in ~3.5-4 hours on a T4 GPU, well within Kaggle's 12-hour limit.
import json, glob, random, os
from pathlib import Path

sample_dir = Path('/kaggle/temp/agar_sample')
sample_dir.mkdir(parents=True, exist_ok=True)

all_jsons = glob.glob(f'{agar_dir}/*.json') + glob.glob(f'{agar_dir}/**/*.json', recursive=True)
all_jsons = [p for p in all_jsons if not p.endswith('annotations.json')]

countable = []
empty = []

for jp in all_jsons:
    stem = Path(jp).stem
    img = Path(jp).parent / f'{stem}.jpg'
    if not img.exists():
        continue
    try:
        with open(jp, 'r') as f:
            d = json.load(f)
        n = d.get('colonies_number', len(d.get('labels', [])))
        if n == 0:
            empty.append((img, jp))
        elif 1 <= n <= 300:
            countable.append((img, jp))
    except Exception:
        continue

random.seed(42)
selected = random.sample(countable, min(len(countable), 1400)) + random.sample(empty, min(len(empty), 100))
print(f'Selected {len(selected)} plates ({len(countable)} countable, {len(empty)} empty found)')

for img, jp in selected:
    dest_img = sample_dir / Path(img).name
    dest_json = sample_dir / Path(jp).name
    if not dest_img.exists():
        os.symlink(img, dest_img)
    if not dest_json.exists():
        os.symlink(jp, dest_json)

# Slice the selected plates into 640x640 native-resolution tiles
!python tools/build_training_set.py \
    --dataset /kaggle/temp/agar_sample \
    --out /kaggle/temp/tiles \
    --scale 1.0 \
    --empty-keep 0.15 \
    --coco \
    --gold "" 2>&1 | grep -E "tiles:|boxes:|plates|Error|Traceback" | tee {OUT}/tile_log.txt

!ls -la /kaggle/temp/tiles/coco

In [ ]:
# 4. Train RF-DETR Medium on Plate Tiles
# ~20,000 tiles takes ~20 min/epoch on T4 GPU; 10 epochs completes in ~3.5-4 hours.
# Checkpoints are saved directly to /kaggle/working/rfdetr so every epoch is immediately preserved.
!python tools/train_rfdetr.py \
    --dataset /kaggle/temp/tiles/coco \
    --size medium \
    --resolution 640 \
    --epochs 10 \
    --batch 4 \
    --grad-accum 4 \
    --lr 1e-4 \
    --output {OUT}/rfdetr \
    --out {OUT}/rfdetr_tiles.pth \
    2>&1 | grep -E --line-buffered "Epoch|Best|mAP|Error|Traceback|->" | tee {OUT}/train_log.txt

print('Training complete! Best checkpoint copied to', f'{OUT}/rfdetr_tiles.pth')

In [ ]:
# 5. Tune Confidence Cut-offs on Validation Plates
os.environ['CD_RFDETR_WEIGHTS'] = f'{OUT}/rfdetr_tiles.pth'
os.environ['CD_ENGINES'] = 'rfdetr'

!python tools/tune_detector.py \
    --dataset /kaggle/temp/agar_sample \
    --engine rfdetr \
    --split valid \
    --conf-only \
    --cache /kaggle/temp/valid_dets.pkl | tee {OUT}/tune_log.txt

!cp models/detector.json {OUT}/detector.json
print('\nFinal generated files in output:')
!ls -lh {OUT}